# snake-rl — results notebook (Part E)

Regenerates every table and figure of the report from the training logs. `snake_rl.analysis.report` reads the run directories under `runs/` (per seed: `config.yaml`, `metrics.csv`, `final_eval.json`), aggregates the five seeds of every configuration with Student-t 95 % intervals (the seed is the unit of replication) and writes `runs/report/`: tables as Markdown, figures as PNG, `index.md` and `manifest.json`. No number of the report is typed by hand; these files are the numbers.

The logs are not in the repository (`runs/` is about 1 GB with checkpoints; the logs alone are about 12 MB). Three ways to get them into this session:

1. upload `runs_logs.zip`, made with `python scripts/export_logs.py` on the machine that trained, when the cell below asks for it;
2. set `LOGS_URL` to a link to that zip (for example a GitHub release asset);
3. run the notebook inside a checkout that already has `runs/`.

Behaviour analyses (heat-maps, loop statistics, hacking recordings) need the weights of a run, which the zip does not carry; the last cell runs one when a `model.pt` is present.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Cureeeeeeee/Value-Based-Deep-Reinforcement-Learning-on-Snake.git"
if not os.path.isdir("snake_rl"):  # not inside a checkout: clone one (Colab)
    if not os.path.isdir("snake-rl"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "snake-rl"], check=True)
    os.chdir("snake-rl")
%pip install -q -e .
print("working directory:", os.getcwd())

In [ ]:
import glob
import urllib.request
import zipfile

LOGS_URL = ""               # a link to runs_logs.zip (scripts/export_logs.py); empty: upload or local runs/
LOGS_ZIP = "runs_logs.zip"


def experiment_dirs():
    return sorted(d for d in glob.glob("runs/*/") if glob.glob(os.path.join(d, "*", "final_eval.json")))


if not experiment_dirs():
    if LOGS_URL:
        urllib.request.urlretrieve(LOGS_URL, LOGS_ZIP)
    elif not os.path.exists(LOGS_ZIP):
        try:
            from google.colab import files  # Colab: pick runs_logs.zip in the upload dialog
        except ImportError:
            raise SystemExit("no runs/ and no runs_logs.zip: set LOGS_URL or copy the zip next to this notebook")
        uploaded = files.upload()
        LOGS_ZIP = next(iter(uploaded))
    with zipfile.ZipFile(LOGS_ZIP) as zf:
        zf.extractall(".")  # entries are runs/<experiment>/<seed>/<file>
print(len(experiment_dirs()), "experiment directories with finished runs under runs/")

In [ ]:
# What the logs hold against the experiment matrix (snake_rl/analysis/matrix.py):
# finished seeds per configuration, and whether every recorded config.yaml is the
# configuration the matrix says.
!python -m snake_rl.analysis.matrix --status

In [ ]:
# Every table and figure of the report, from the logs; items whose experiments are
# missing are skipped and listed in runs/report/index.md.
!python -m snake_rl.analysis.report --runs-dir runs --out-dir runs/report

In [ ]:
import json

from IPython.display import Image, Markdown, display

with open("runs/report/manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)
print("code", manifest["git"], "| last-k column:", manifest["last_k"])
for item in manifest["items"]:
    if not item["outputs"]:
        continue
    display(Markdown(f"## {item['name']}\n\n{item.get('title') or ''}"))
    for kind, rel in item["outputs"].items():
        path = os.path.join("runs/report", rel)
        if kind == "table":
            with open(path, encoding="utf-8") as f:
                display(Markdown(f.read()))
        else:
            display(Image(filename=path))

## Behaviour analysis (needs weights)

`snake_rl.analysis.behaviour` plays 100 evaluation episodes with a trained network and writes the visit heat-map, the loop / shaping statistics per episode and GUI-replayable recordings of the longest loops. It needs a `model.pt`, which the logs zip does not carry; the cell runs when one is present (train a run with `python -m snake_rl.train --config configs/dqn_B_8x8.yaml --seed 0 --set agent.gamma=0.99 --exp dqn_B8_W8_gamma099`, or copy a run directory here).

In [ ]:
RUN_DIR = "runs/dqn_B8_W8_gamma099/0"  # any run directory with model.pt
OUT_DIR = "runs/behaviour/notebook"
if os.path.exists(os.path.join(RUN_DIR, "model.pt")):
    !python -m snake_rl.analysis.behaviour --player {RUN_DIR} --checkpoint final --out-dir {OUT_DIR}
    display(Image(filename=os.path.join(OUT_DIR, "heatmap.png")))
    with open(os.path.join(OUT_DIR, "summary.json"), encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2)[:2000])
else:
    print("no weights at", RUN_DIR, "- the logs zip carries none; see the note above")